In [1]:
import pandas as pd
import numpy as np

from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.linear_model import LinearRegression, ElasticNet
from sklearn.pipeline import Pipeline
from sklearn.metrics import mean_absolute_percentage_error
from sklearn.svm import SVR
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from xgboost import XGBRegressor

import mlflow

In [2]:
# set the dagshub tracking server

mlflow.set_tracking_uri("https://dagshub.com/aditya14-sagar/Uber-Demand-Prediction.mlflow")

In [3]:
import dagshub
dagshub.init(repo_owner='aditya14-sagar', repo_name='Uber-Demand-Prediction', mlflow=True)

Accessing as aditya14-sagar

Initialized MLflow to track repo "aditya14-sagar/Uber-Demand-Prediction"

Repository aditya14-sagar/Uber-Demand-Prediction initialized!

In [4]:
# load the training and test data

train_data_path = "../data/processed/train.csv"
test_data_path = "../data/processed/test.csv"

train_df = pd.read_csv(train_data_path, parse_dates=["tpep_pickup_datetime"]).set_index("tpep_pickup_datetime")

test_df = pd.read_csv(test_data_path, parse_dates=["tpep_pickup_datetime"]).set_index("tpep_pickup_datetime")

train_df

,lag_1,lag_2,lag_3,lag_4,region,total_pickups,avg_pickups,day_of_week
tpep_pickup_datetime,,,,,,,,
2016-01-01 01:00:00,140.0,143.0,107.0,46.0,0,179,126.0,4
2016-01-01 01:15:00,179.0,140.0,143.0,107.0,0,189,149.0,4
2016-01-01 01:30:00,189.0,179.0,140.0,143.0,0,163,166.0,4
2016-01-01 01:45:00,163.0,189.0,179.0,140.0,0,187,165.0,4
2016-01-01 02:00:00,187.0,163.0,189.0,179.0,0,170,174.0,4
...,...,...,...,...,...,...,...,...
2016-02-29 22:45:00,57.0,76.0,56.0,55.0,23,48,63.0,0
2016-02-29 23:00:00,48.0,57.0,76.0,56.0,23,40,57.0,0
2016-02-29 23:15:00,40.0,48.0,57.0,76.0,23,26,50.0,0


In [5]:
# missing value in training data

train_df.isna().sum()

lag_1            0
lag_2            0
lag_3            0
lag_4            0
region           0
total_pickups    0
avg_pickups      0
day_of_week      0
dtype: int64

In [6]:
# missing values in the test data

test_df.isna().sum()

lag_1            0
lag_2            0
lag_3            0
lag_4            0
region           0
total_pickups    0
avg_pickups      0
day_of_week      0
dtype: int64

In [7]:
# make X_train and y_train

X_train = train_df.drop(columns=["total_pickups"])

y_train = train_df["total_pickups"]

In [8]:
X_train.head()

,lag_1,lag_2,lag_3,lag_4,region,avg_pickups,day_of_week
tpep_pickup_datetime,,,,,,,
2016-01-01 01:00:00,140.0,143.0,107.0,46.0,0,126.0,4
2016-01-01 01:15:00,179.0,140.0,143.0,107.0,0,149.0,4
2016-01-01 01:30:00,189.0,179.0,140.0,143.0,0,166.0,4
2016-01-01 01:45:00,163.0,189.0,179.0,140.0,0,165.0,4
2016-01-01 02:00:00,187.0,163.0,189.0,179.0,0,174.0,4


In [9]:
# make X_test and y_test

X_test = test_df.drop(columns=["total_pickups"])

y_test = test_df["total_pickups"]

In [10]:
X_test.head()

,lag_1,lag_2,lag_3,lag_4,region,avg_pickups,day_of_week
tpep_pickup_datetime,,,,,,,
2016-03-01 00:00:00,32.0,38.0,30.0,24.0,0,33.0,1
2016-03-01 00:15:00,33.0,32.0,38.0,30.0,0,33.0,1
2016-03-01 00:30:00,29.0,33.0,32.0,38.0,0,31.0,1
2016-03-01 00:45:00,37.0,29.0,33.0,32.0,0,34.0,1
2016-03-01 01:00:00,29.0,37.0,29.0,33.0,0,32.0,1


In [11]:
from sklearn import set_config

set_config(transform_output="pandas")

In [12]:
encoder = ColumnTransformer([
    ("ohe", OneHotEncoder(drop="first", sparse_output=False), ["region", "day_of_week"])
], remainder="passthrough", n_jobs=-1)

In [13]:
encoder

ColumnTransformer(n_jobs=-1, remainder='passthrough',
                  transformers=[('ohe',
                                 OneHotEncoder(drop='first',
                                               sparse_output=False),
                                 ['region', 'day_of_week'])])

In [14]:
# encode the train and test data

X_train_encoded = encoder.fit_transform(X_train)
X_test_encoded = encoder.transform(X_test)

In [15]:
import optuna

d:\Uber-Demand-Prediction\myenv\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [16]:
# set the experiment

mlflow.set_experiment("Model Selection")

<Experiment: artifact_location='mlflow-artifacts:/dbac01cf12fe4b1f913a4f79ea256d42', creation_time=1780043686399, experiment_id='0', last_update_time=1780043686399, lifecycle_stage='active', name='Model Selection', tags={'mlflow.experimentKind': 'custom_model_development'}>

In [17]:
from sklearn.ensemble import HistGradientBoostingRegressor



def objective(trial):
    # start the child run
    with mlflow.start_run(nested=True) as child:
        
        # model name search space
        list_of_models = ["LR", "RF", "GBR", "XGBR"]
        model_name = trial.suggest_categorical("model_name", list_of_models)
    
        if model_name == "LR":
            alpha_lr = trial.suggest_float("alpha_lr", 1e-4, 10.0, log=True)
            l1_ratio_lr = trial.suggest_float("l1_ratio_lr", 0.0, 1.0)
            model = ElasticNet(alpha=alpha_lr, l1_ratio=l1_ratio_lr, random_state=42)
            
        elif model_name == "RF":
            n_estimators_rf = trial.suggest_int("n_estimators_rf", 50, 300, step=25)
            max_depth_rf = trial.suggest_int("max_depth_rf", 3, 20)
            min_samples_leaf_rf = trial.suggest_int("min_samples_leaf_rf", 1, 10)
            max_features_rf = trial.suggest_float("max_features_rf", 0.3, 1.0)
            model = RandomForestRegressor(
                n_estimators=n_estimators_rf,
                max_depth=max_depth_rf,
                min_samples_leaf=min_samples_leaf_rf,
                max_features=max_features_rf,
                random_state=42, n_jobs=-1
            )
    
        elif model_name == "GBR":
            max_iter_gb = trial.suggest_int("max_iter_gb", 50, 300, step=25)
            learning_rate_gb = trial.suggest_float("learning_rate_gb", 1e-3, 3e-1, log=True)
            max_depth_gb = trial.suggest_int("max_depth_gb", 2, 10)
            model = HistGradientBoostingRegressor(
                max_iter=max_iter_gb,
                learning_rate=learning_rate_gb,
                max_depth=max_depth_gb,
                random_state=42
            )

        elif model_name == "XGBR":
            n_estimators_xgb = trial.suggest_int("n_estimators_xgb", 50, 500, step=25)
            learning_rate_xgb = trial.suggest_float("learning_rate_xgb", 1e-3, 3e-1, log=True)
            max_depth_xgb = trial.suggest_int("max_depth_xgb", 3, 10)
            subsample_xgb = trial.suggest_float("subsample_xgb", 0.5, 1.0)
            colsample_bytree_xgb = trial.suggest_float("colsample_bytree_xgb", 0.5, 1.0)
            reg_alpha_xgb = trial.suggest_float("reg_alpha_xgb", 1e-8, 10.0, log=True)
            reg_lambda_xgb = trial.suggest_float("reg_lambda_xgb", 1e-8, 10.0, log=True)
            min_child_weight_xgb = trial.suggest_int("min_child_weight_xgb", 1, 10)
            model = XGBRegressor(
                n_estimators=n_estimators_xgb,
                learning_rate=learning_rate_xgb,
                max_depth=max_depth_xgb,
                subsample=subsample_xgb,
                colsample_bytree=colsample_bytree_xgb,
                reg_alpha=reg_alpha_xgb,
                reg_lambda=reg_lambda_xgb,
                min_child_weight=min_child_weight_xgb,
                random_state=42, n_jobs=-1
            )
    
        # log the model name
        mlflow.log_param("model_name",model_name)
        
        # log the model parameters
        mlflow.log_params(model.get_params())
        
        # fit on the data
        model.fit(X_train_encoded,y_train)
    
        # get the predictions
        y_pred = model.predict(X_test_encoded)
    
        # calculate the loss
        loss = mean_absolute_percentage_error(y_test, y_pred)
    
        # log the metric
        mlflow.log_metric("MAPE",loss)
        return loss

In [18]:
# optimize the objective function

with mlflow.start_run(run_name="best_model", nested=True) as parent:

    # create a study object
    study = optuna.create_study(study_name="model_selection", direction="minimize")
    # optimize the objective function
    study.optimize(func=objective, n_trials=35, n_jobs=1)
    
    # log the best parameters
    mlflow.log_params(study.best_params)
    # log the best error value
    mlflow.log_metric("Best_MAPE", study.best_value)

[I 2026-07-24 17:57:32,493] A new study created in memory with name: model_selection
2026/07/24 17:57:36 INFO mlflow.tracking._tracking_service.client: 🏃 View run youthful-goat-140 at: https://dagshub.com/aditya14-sagar/Uber-Demand-Prediction.mlflow/#/experiments/0/runs/f4f4d695eeda47e8987e2775377fe1d5.
2026/07/24 17:57:36 INFO mlflow.tracking._tracking_service.client: 🧪 View experiment at: https://dagshub.com/aditya14-sagar/Uber-Demand-Prediction.mlflow/#/experiments/0.
[I 2026-07-24 17:57:36,639] Trial 0 finished with value: 0.26250737782427336 and parameters: {'model_name': 'RF', 'n_estimators_rf': 50, 'max_depth_rf': 10, 'min_samples_leaf_rf': 3, 'max_features_rf': 0.4131330168257165}. Best is trial 0 with value: 0.26250737782427336.
d:\Uber-Demand-Prediction\myenv\lib\site-packages\sklearn\linear_model\_coordinate_descent.py:631: ConvergenceWarning: Objective did not converge. You might want to increase the number of iterations, check the scale of the features or consider increasi

In [19]:
# best value

study.best_value

0.23830235811154984

In [20]:
# best parameters

study.best_params

{'model_name': 'XGBR',
 'n_estimators_xgb': 450,
 'learning_rate_xgb': 0.14436703603473178,
 'max_depth_xgb': 10,
 'subsample_xgb': 0.896026783990372,
 'colsample_bytree_xgb': 0.991222460111412,
 'reg_alpha_xgb': 7.346721322483371e-08,
 'reg_lambda_xgb': 0.1343831593157791,
 'min_child_weight_xgb': 9}

In [21]:
# model value counts

study.trials_dataframe()['params_model_name'].value_counts()

XGBR    22
RF       5
LR       4
GBR      4
Name: params_model_name, dtype: int64

In [22]:
from optuna.visualization import (
    plot_optimization_history, 
    plot_parallel_coordinate, 
    plot_param_importances
)

In [23]:
import plotly
plot_optimization_history(study)

In [24]:
plot_parallel_coordinate(study, params=["model_name"])